# 04.05 - Cross-Validation: Validación Robusta de Modelos

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-04-predictivo-proyecto/notebooks/04_05_cross_validation.ipynb)

## Objetivos de Aprendizaje

Al finalizar este notebook, usted será capaz de:

1. **Comprender** por qué una sola división train/test puede ser insuficiente
2. **Implementar** K-Fold Cross-Validation
3. **Interpretar** resultados de validación cruzada
4. **Seleccionar** modelos de manera más robusta

## Duración estimada: 60 minutos

---

## 1. El Problema de Una Sola División

### 1.1 ¿Por qué train/test no es suficiente?

Hasta ahora hemos dividido datos en train (80%) y test (20%). Pero esto tiene problemas:

1. **Variabilidad**: Diferentes divisiones aleatorias dan diferentes resultados
2. **Desperdicio**: El 20% de test nunca se usa para entrenar
3. **Incertidumbre**: No sabemos qué tan estable es nuestra métrica

### 1.2 Demostración del Problema

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, KFold, StratifiedKFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, r2_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas")

In [ ]:
# Cargar datos
url = "https://raw.githubusercontent.com/gonzalezulises/formacion-docente-bi-faces/main/datasets/universidad/rendimiento_academico.csv"
df = pd.read_csv(url)

# Preparar datos para clasificación binaria
df['aprobado'] = (df['promedio'] >= 10).astype(int)

# Seleccionar features numéricas
features = ['asistencia', 'horas_estudio', 'materias_inscritas']
X = df[features].dropna()
y = df.loc[X.index, 'aprobado']

print(f"Datos: {len(X)} observaciones, {len(features)} features")
print(f"Distribución de clases: {y.value_counts(normalize=True).to_dict()}")

In [ ]:
# Demostración: diferentes divisiones dan diferentes resultados
resultados = []

for seed in range(20):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )
    
    modelo = LogisticRegression(random_state=42)
    modelo.fit(X_train, y_train)
    
    accuracy = accuracy_score(y_test, modelo.predict(X_test))
    resultados.append(accuracy)

print(f"Accuracy con diferentes divisiones:")
print(f"  Mínimo: {min(resultados):.3f}")
print(f"  Máximo: {max(resultados):.3f}")
print(f"  Rango: {max(resultados) - min(resultados):.3f}")
print(f"  Promedio: {np.mean(resultados):.3f}")
print(f"  Desv. Est: {np.std(resultados):.3f}")

In [ ]:
# Visualizar la variabilidad
plt.figure(figsize=(10, 5))
plt.bar(range(20), resultados, color='steelblue', alpha=0.7)
plt.axhline(y=np.mean(resultados), color='red', linestyle='--', label=f'Promedio: {np.mean(resultados):.3f}')
plt.xlabel('Semilla aleatoria')
plt.ylabel('Accuracy')
plt.title('Variabilidad del Accuracy según la división train/test')
plt.legend()
plt.tight_layout()
plt.show()

print("\n⚠️ Problema: El accuracy varía significativamente según cómo dividimos los datos")

---

## 2. K-Fold Cross-Validation

### 2.1 La Idea

En lugar de una sola división, hacemos **K divisiones** y promediamos:

```
K-Fold Cross-Validation (K=5):

Iteración 1: [TEST][Train][Train][Train][Train]
Iteración 2: [Train][TEST][Train][Train][Train]
Iteración 3: [Train][Train][TEST][Train][Train]
Iteración 4: [Train][Train][Train][TEST][Train]
Iteración 5: [Train][Train][Train][Train][TEST]

→ Promedio de 5 evaluaciones
```

### 2.2 Ventajas

1. **Usa todos los datos** tanto para entrenamiento como para prueba
2. **Reduce varianza** del estimador de error
3. **Provee intervalo** de confianza para el desempeño
4. **Detecta overfitting** si hay mucha varianza entre folds

In [ ]:
# Implementación básica de K-Fold
from sklearn.model_selection import cross_val_score

modelo = LogisticRegression(random_state=42)

# Cross-validation con 5 folds
scores = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')

print("=== Cross-Validation con 5 Folds ===")
print(f"\nScores por fold: {scores.round(3)}")
print(f"\nPromedio: {scores.mean():.3f}")
print(f"Desv. Est: {scores.std():.3f}")
print(f"\nIntervalo de confianza (95%): {scores.mean():.3f} ± {scores.std()*2:.3f}")

### 2.3 Visualización del Proceso

In [ ]:
# Visualizar el proceso de K-Fold
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

fig, axes = plt.subplots(5, 1, figsize=(12, 6))

for i, (train_idx, test_idx) in enumerate(kf.split(X)):
    # Crear array para visualizar
    fold_visual = np.zeros(len(X))
    fold_visual[train_idx] = 1  # Train = 1
    fold_visual[test_idx] = 2   # Test = 2
    
    axes[i].imshow([fold_visual], aspect='auto', cmap='RdYlGn', vmin=0, vmax=2)
    axes[i].set_yticks([])
    axes[i].set_ylabel(f'Fold {i+1}')
    
axes[0].set_title('Visualización de K-Fold CV (Verde=Train, Rojo=Test)')
axes[-1].set_xlabel('Índice de observación')
plt.tight_layout()
plt.show()

---

## 3. Variantes de Cross-Validation

### 3.1 Stratified K-Fold

**Problema**: En clasificación con clases desbalanceadas, algunos folds pueden tener proporciones muy diferentes.

**Solución**: Stratified K-Fold mantiene la proporción de clases en cada fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Comparar KFold normal vs Stratified
kf_normal = KFold(n_splits=5, shuffle=True, random_state=42)
kf_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=== Comparación de proporciones de clase por fold ===")
print("\nKFold Normal:")
for i, (train_idx, test_idx) in enumerate(kf_normal.split(X, y)):
    prop = y.iloc[test_idx].mean()
    print(f"  Fold {i+1}: {prop:.3f} (clase 1)")

print("\nStratified KFold:")
for i, (train_idx, test_idx) in enumerate(kf_strat.split(X, y)):
    prop = y.iloc[test_idx].mean()
    print(f"  Fold {i+1}: {prop:.3f} (clase 1)")

print(f"\nProporción original: {y.mean():.3f}")

In [ ]:
# En la práctica, usar stratified para clasificación
scores_strat = cross_val_score(
    LogisticRegression(random_state=42), 
    X, y, 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy'
)

print(f"Stratified K-Fold: {scores_strat.mean():.3f} ± {scores_strat.std():.3f}")

### 3.2 Leave-One-Out (LOO)

Caso extremo donde K = N (número de observaciones). Cada observación es test una vez.

- **Ventaja**: Máximo uso de datos para entrenamiento
- **Desventaja**: Computacionalmente costoso, alta varianza

In [ ]:
from sklearn.model_selection import LeaveOneOut

# Solo demostración con subset pequeño
X_small = X.iloc[:100]
y_small = y.iloc[:100]

loo = LeaveOneOut()
scores_loo = cross_val_score(
    LogisticRegression(random_state=42),
    X_small, y_small,
    cv=loo,
    scoring='accuracy'
)

print(f"Leave-One-Out (n=100): {scores_loo.mean():.3f}")
print(f"Número de iteraciones: {len(scores_loo)}")

### 3.3 Repeated K-Fold

Repite K-Fold múltiples veces con diferentes divisiones aleatorias.

In [ ]:
from sklearn.model_selection import RepeatedStratifiedKFold

# 5-fold repetido 3 veces = 15 evaluaciones
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

scores_repeated = cross_val_score(
    LogisticRegression(random_state=42),
    X, y,
    cv=rskf,
    scoring='accuracy'
)

print(f"Repeated Stratified K-Fold (5 folds × 3 repeticiones):")
print(f"  Promedio: {scores_repeated.mean():.3f}")
print(f"  Desv. Est: {scores_repeated.std():.3f}")
print(f"  Número de evaluaciones: {len(scores_repeated)}")

---

## 4. Selección de Modelos con Cross-Validation

Un uso importante de CV es comparar modelos de manera justa.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Definir modelos a comparar
modelos = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
}

# Evaluar cada modelo con CV
resultados_cv = {}

for nombre, modelo in modelos.items():
    scores = cross_val_score(
        modelo, X, y,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='accuracy'
    )
    resultados_cv[nombre] = {
        'mean': scores.mean(),
        'std': scores.std(),
        'scores': scores
    }
    print(f"{nombre:25s}: {scores.mean():.3f} ± {scores.std():.3f}")

In [ ]:
# Visualizar comparación
fig, ax = plt.subplots(figsize=(10, 6))

nombres = list(resultados_cv.keys())
means = [resultados_cv[n]['mean'] for n in nombres]
stds = [resultados_cv[n]['std'] for n in nombres]

# Ordenar por media
orden = np.argsort(means)[::-1]
nombres = [nombres[i] for i in orden]
means = [means[i] for i in orden]
stds = [stds[i] for i in orden]

bars = ax.barh(nombres, means, xerr=stds, capsize=5, color='steelblue', alpha=0.7)
ax.set_xlabel('Accuracy (con intervalo de confianza)')
ax.set_title('Comparación de Modelos con Cross-Validation')

# Añadir valores
for bar, mean, std in zip(bars, means, stds):
    ax.text(mean + std + 0.01, bar.get_y() + bar.get_height()/2,
            f'{mean:.3f}±{std:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

---

## 5. Cross-Validation con Múltiples Métricas

In [ ]:
from sklearn.model_selection import cross_validate

# Evaluar con múltiples métricas
scoring = ['accuracy', 'precision', 'recall', 'f1']

modelo = RandomForestClassifier(n_estimators=100, random_state=42)

resultados_multi = cross_validate(
    modelo, X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=scoring,
    return_train_score=True
)

print("=== Random Forest: Múltiples Métricas ===")
print("\nMétrica          | Train      | Test")
print("-" * 45)
for metrica in scoring:
    train_mean = resultados_multi[f'train_{metrica}'].mean()
    test_mean = resultados_multi[f'test_{metrica}'].mean()
    print(f"{metrica:16s} | {train_mean:.3f}      | {test_mean:.3f}")

### Detectar Overfitting

Si train >> test, hay overfitting.

In [ ]:
# Comparar train vs test para detectar overfitting
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(scoring))
width = 0.35

train_means = [resultados_multi[f'train_{m}'].mean() for m in scoring]
test_means = [resultados_multi[f'test_{m}'].mean() for m in scoring]

ax.bar(x - width/2, train_means, width, label='Train', color='lightblue')
ax.bar(x + width/2, test_means, width, label='Test', color='steelblue')

ax.set_ylabel('Score')
ax.set_title('Comparación Train vs Test (detectar overfitting)')
ax.set_xticks(x)
ax.set_xticklabels(scoring)
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print("\n💡 Si las barras de Train son mucho más altas que Test, hay overfitting")

---

## 6. Buenas Prácticas

### 6.1 ¿Cuántos folds usar?

| K | Pros | Contras | Cuándo usar |
|---|------|---------|-------------|
| 5 | Balance tiempo/varianza | Estándar general | Default recomendado |
| 10 | Menor sesgo | Más varianza, más lento | Datasets medianos-grandes |
| N (LOO) | Mínimo sesgo | Alta varianza, muy lento | Datasets muy pequeños |

### 6.2 Errores Comunes

1. **Data leakage**: Preprocesar antes de dividir (ej: normalizar con todo el dataset)
2. **No usar stratified** para clasificación desbalanceada
3. **Seleccionar hiperparámetros con CV y reportar ese mismo score** (necesita CV anidado)

### 6.3 Protocolo Correcto

```
1. Reservar test set final (holdout)
2. Usar CV en el resto para:
   - Comparar modelos
   - Seleccionar hiperparámetros
3. Entrenar modelo final con todos los datos de train
4. Evaluar UNA VEZ en test set
```

---

## 7. Ejercicio Práctico

Use cross-validation para seleccionar el mejor modelo para predecir rendimiento académico.

In [ ]:
# EJERCICIO: Complete el código

# 1. Defina al menos 3 modelos diferentes
# 2. Evalúe cada uno con 5-fold stratified CV
# 3. Reporte accuracy, precision, recall para cada uno
# 4. Seleccione el mejor y justifique

# Su código aquí:
# ...

---

## Resumen

### Conceptos Clave

1. **Una sola división train/test es inestable**: Los resultados varían según la división

2. **K-Fold CV** divide los datos en K partes y entrena K veces, usando cada parte como test una vez

3. **Stratified K-Fold** mantiene proporciones de clases en cada fold

4. **Ventajas de CV**:
   - Usa todos los datos para train y test
   - Reduce varianza del estimador
   - Detecta overfitting
   - Permite comparación justa de modelos

### Para su Proyecto Integrador

- Use 5-fold stratified CV para reportar métricas
- Reporte media ± desviación estándar
- Compare train vs test para verificar overfitting

---

**Próximo paso**: Use CV en su proyecto integrador para seleccionar y evaluar modelos.